# Tratamento de Dados do YouTube com PySpark

Neste notebook realizamos a limpeza, transformação e integração dos dados de vídeos, comentários e vídeos em tendência do YouTube, preparando uma base sólida para análises posteriores.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DateType

spark = SparkSession.builder \
    .appName('YouTube - Tratamento de Dados') \
    .config('spark.sql.warehouse.dir', '/tmp/spark-warehouse') \
    .getOrCreate()

spark.sparkContext.setLogLevel('ERROR')
print(f'SparkSession iniciada | Spark {spark.version}')

## Passo 1 — Leitura do `videos-stats.csv` com inferência de esquema

In [ ]:
# Passo 1: Leitura do arquivo videos-stats.csv com cabeçalho e inferência de esquema
df_video = spark.read \
    .option('header', 'true') \
    .option('inferSchema', 'true') \
    .csv('videos-stats.csv')

print('Esquema do df_video:')
df_video.printSchema()
df_video.show(5, truncate=True)

## Passo 2 — Substituição de valores nulos por 0 em `Likes`, `Comments` e `Views`

In [ ]:
# Passo 2: Alterar valores nulos dos campos Likes, Comments e Views para 0
df_video = df_video.fillna(0, subset=['Likes', 'Comments', 'Views'])

print('Valores nulos em Likes, Comments e Views substituídos por 0.')
df_video.select('Likes', 'Comments', 'Views').show(5)

## Passo 3 — Leitura do `comments.csv` com inferência de esquema

In [ ]:
# Passo 3: Leitura do arquivo comments.csv com cabeçalho e inferência de esquema
# Usamos multiLine=true para lidar com comentários que contêm quebras de linha internas
df_comentario = spark.read \
    .option('header', 'true') \
    .option('inferSchema', 'true') \
    .option('multiLine', 'true') \
    .option('escape', '"') \
    .csv('comments.csv')

print('Esquema do df_comentario:')
df_comentario.printSchema()
df_comentario.show(5, truncate=True)

## Passo 4 — Quantidade de registros em `df_video` e `df_comentario`

In [ ]:
# Passo 4: Calcular a quantidade de registros de cada dataframe
qtd_video = df_video.count()
qtd_comentario = df_comentario.count()

print(f'Quantidade de registros em df_video:      {qtd_video}')
print(f'Quantidade de registros em df_comentario: {qtd_comentario}')

## Passo 5 — Remoção de registros com `Video ID` nulos

In [ ]:
# Passo 5: Remover registros com campo 'Video ID' nulo e recalcular contagens
df_video = df_video.filter(F.col('Video ID').isNotNull())
df_comentario = df_comentario.filter(F.col('Video ID').isNotNull())

qtd_video_limpo = df_video.count()
qtd_comentario_limpo = df_comentario.count()

print(f'df_video após remover nulos em Video ID:      {qtd_video_limpo} registros')
print(f'df_comentario após remover nulos em Video ID: {qtd_comentario_limpo} registros')

## Passo 6 — Remoção de `Video ID` duplicados em `df_video`

In [ ]:
# Passo 6: Remover registros com 'Video ID' duplicados apenas no df_video
qtd_antes = df_video.count()
df_video = df_video.dropDuplicates(['Video ID'])
qtd_depois = df_video.count()

print(f'Registros antes da remoção de duplicatas: {qtd_antes}')
print(f'Registros após a remoção de duplicatas:   {qtd_depois}')
print(f'Duplicatas removidas:                     {qtd_antes - qtd_depois}')

## Passo 7 — Conversão de `Likes`, `Comments` e `Views` para `int` em `df_video`

In [ ]:
# Passo 7: Converter Likes, Comments e Views para inteiro no df_video
# Usamos LongType para Views pois o valor máximo (~4 bilhões) excede o limite do IntegerType
from pyspark.sql.types import LongType
df_video = df_video \
    .withColumn('Likes',    F.col('Likes').cast('double').cast(IntegerType())) \
    .withColumn('Comments', F.col('Comments').cast('double').cast(IntegerType())) \
    .withColumn('Views',    F.col('Views').cast('double').cast(LongType()))

print('Tipos após conversão:')
df_video.select('Likes', 'Comments', 'Views').printSchema()

## Passo 8 — Conversão de `Likes` e `Sentiment` para `int` em `df_comentario` e renomeação de `Likes`

In [ ]:
# Passo 8: Converter Likes e Sentiment para int e renomear Likes para 'Likes Comment'
# Os valores estão em formato float (ex: 95.0), então convertemos via double antes de int
df_comentario = df_comentario \
    .withColumn('Likes',     F.col('Likes').cast('double').cast(IntegerType())) \
    .withColumn('Sentiment', F.col('Sentiment').cast('double').cast(IntegerType())) \
    .withColumnRenamed('Likes', 'Likes Comment')

print('Esquema do df_comentario após transformações:')
df_comentario.printSchema()
df_comentario.show(5, truncate=True)

## Passo 9 — Criação do campo `Interaction` em `df_video`

In [ ]:
# Passo 9: Criar campo 'Interaction' com a soma de Likes + Comments + Views
df_video = df_video.withColumn(
    'Interaction',
    F.col('Likes') + F.col('Comments') + F.col('Views')
)

print('Campo Interaction criado:')
df_video.select('Likes', 'Comments', 'Views', 'Interaction').show(5)

## Passo 10 — Conversão do campo `Published At` para `date`

In [ ]:
# Passo 10: Converter 'Published At' para o tipo date
df_video = df_video.withColumn(
    'Published At',
    F.to_date(F.col('Published At'), 'yyyy-MM-dd')
)

print('Campo Published At convertido para date:')
df_video.select('Published At').printSchema()
df_video.select('Published At').show(5)

## Passo 11 — Criação do campo `Year` extraído de `Published At`

In [ ]:
# Passo 11: Criar campo 'Year' extraindo o ano de 'Published At'
df_video = df_video.withColumn(
    'Year',
    F.year(F.col('Published At'))
)

print('Campo Year criado:')
df_video.select('Published At', 'Year').show(5)

## Passo 12 — Join de `df_video` com `df_comentario` pelo campo `Video ID`

In [ ]:
# Passo 12: Mesclar df_comentario no df_video pelo campo 'Video ID' (left join)
df_join_video_comments = df_video.join(
    df_comentario,
    on='Video ID',
    how='left'
)

print('Esquema do df_join_video_comments:')
df_join_video_comments.printSchema()

print(f'Total de registros: {df_join_video_comments.count()}')
df_join_video_comments.show(5, truncate=True)

## Passo 13 — Leitura do arquivo `USvideos.csv`

In [ ]:
# Passo 13: Leitura do arquivo USvideos.csv com cabeçalho e inferência de esquema
df_us_videos = spark.read \
    .option('header', 'true') \
    .option('inferSchema', 'true') \
    .option('multiLine', 'true') \
    .option('escape', '"') \
    .csv('USvideos.csv')

print('Esquema do df_us_videos:')
df_us_videos.printSchema()
df_us_videos.show(5, truncate=True)

## Passo 14 — Join de `df_video` com `df_us_videos` pelo campo `Title`

In [ ]:
# Passo 14: Mesclar df_us_videos no df_video pelo campo Title (left join)
# Renomear 'title' do df_us_videos para evitar ambiguidade com 'Title' do df_video
df_us_videos_renamed = df_us_videos.withColumnRenamed('title', 'Title')

df_join_video_usvideos = df_video.join(
    df_us_videos_renamed,
    on='Title',
    how='left'
)

print('Esquema do df_join_video_usvideos:')
df_join_video_usvideos.printSchema()

print(f'Total de registros: {df_join_video_usvideos.count()}')
df_join_video_usvideos.show(5, truncate=True)

## Passo 15 — Verificação de valores nulos em todos os campos de `df_video`

In [ ]:
# Passo 15: Verificar a quantidade de valores nulos em todos os campos do df_video
from pyspark.sql.functions import col, sum as spark_sum, isnan, when

nulos = df_video.select(
    [spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in df_video.columns]
)

print('Contagem de valores nulos por campo em df_video:')
nulos.show(truncate=False)

## Passo 16 — Remoção de `_c0` e salvamento de `df_video` como `videos-tratados-parquet`

In [ ]:
# Passo 16: Remover a coluna '_c0' e salvar df_video como parquet com cabeçalho
df_video = df_video.drop('_c0')

df_video.write \
    .option('header', 'true') \
    .mode('overwrite') \
    .parquet('videos-tratados-parquet')

print('df_video salvo como videos-tratados-parquet com sucesso!')
print(f'Colunas finais: {df_video.columns}')

## Passo 17 — Remoção de `_c0` e salvamento de `df_join_video_comments` como `videos-comments-tratados-parquet`

In [ ]:
# Passo 17: Remover a coluna '_c0' e salvar df_join_video_comments como parquet
# A coluna '_c0' pode aparecer do df_comentario após o join
cols_para_remover = [c for c in df_join_video_comments.columns if c == '_c0']
df_join_video_comments_limpo = df_join_video_comments.drop(*cols_para_remover)

df_join_video_comments_limpo.write \
    .option('header', 'true') \
    .mode('overwrite') \
    .parquet('videos-comments-tratados-parquet')

print('df_join_video_comments salvo como videos-comments-tratados-parquet com sucesso!')
print(f'Colunas finais: {df_join_video_comments_limpo.columns}')

In [ ]:
# Encerramento da SparkSession
spark.stop()
print('SparkSession encerrada com sucesso!')